In [0]:
# Databricks notebook source
# =====================================================
# 1️⃣ Widgets
# =====================================================

import json
from pyspark.sql.functions import current_timestamp

start_time = spark.sql("SELECT current_timestamp()").collect()[0][0]

dbutils.widgets.text(
    "table_metadata",
    "{'table_id': '1', 'table_name': 'customers', 'source_system': 'sqlserver', "
    "'source_schema': 'banking', 'source_table': 'customers', "
    "'source_path': '', 'bronze_schema': 'bronze', "
    "'silver_schema': 'silver', 'active_flag': 'True', "
    "'load_order': '1', 'created_at': '2026-02-18 13:23:37.053711'}"
)

dbutils.widgets.text(
    "table_parameters",
    "{'load_type': 'MERGE', "
    "'primary_key': 'customer_id', "
    "'watermark_column': 'updated_at'}"
)

dbutils.widgets.text(
    "run_id",
    "472519629468310"
)

In [0]:
run_id=dbutils.widgets.get("run_id")

#parse JSON safely
table_metadata=json.loads(dbutils.widgets.get("table_metadata").replace("'",'"'))
table_parameters=json.loads(dbutils.widgets.get("table_parameters").replace("'",'"'))

print("Table Metadata:", table_metadata)
print("Table Parameters:", table_parameters)
print(f"Run ID: {run_id}")

In [0]:

# COMMAND ----------

# =====================================================
# 2️⃣ Extract Variables
# =====================================================

table_id=int(table_metadata['table_id'])
table_name=table_metadata['table_name']
source_system=table_metadata['source_system']
source_schema=table_metadata['source_schema']
source_table=table_metadata['source_table']
source_path=table_metadata['source_path']
bronze_schema=table_metadata['bronze_schema']

load_type=table_parameters.get('load_type')
watermark_column=table_parameters.get('watermark_column')

#create an bronze table
bronze_table_fqn=f"banking.{bronze_schema}.{table_name}"
print(f"Target Bronze Table: {bronze_table_fqn}")

In [0]:
# COMMAND ----------

# DBTITLE 1,Metadata Entry
# =====================================================
# Make and entry to audit table
# =====================================================

entry_exists=spark.sql(f"""
                       SELECT 1
                       FROM banking.metadata.pipeline_runs
                       WHERE run_id={run_id} AND table_id={table_id}
                       """).count()>0

if entry_exists:
    spark.sql(f"""
              UPDATE banking.metadata.pipeline_runs
              SET 
                layer='Silver',
                start_time=TIMESTAMP('{start_time}'),
                end_time=NULL,
                status='INPROGRESS',
                number_of_records=NULL,
                error_message=NULL
              WHERE run_id={run_id} AND table_id={table_id}
              """)
else:
    spark.sql(f""" 
              INSERT INTO banking.metadata.pipeline_runs
              VALUES(
                  {run_id},
                  {table_id},
                  'Silver',
                  TIMESTAMP('{start_time}'),
                  NULL,  --END TIME
                  'INPROGESS',
                  NULL, --NUMBER OF RECORDS
                  NULL --ERROR MESSAGE
              )
              """)


In [0]:
%sql
select * from banking.metadata.pipeline_runs 

In [0]:
# COMMAND ----------

# =====================================================
# 3️⃣ Get Last Watermark (For Filtering Only)
# =====================================================

last_watermark = None

if load_type in ["APPEND", "MERGE"] and watermark_column:
    watermark_df = spark.sql(f"""
        SELECT last_watermark_value
        FROM banking.metadata.table_watermarks
        WHERE table_id = {table_id}
    """)
    
    if watermark_df.count() > 0:
        last_watermark = watermark_df.first()["last_watermark_value"]

print("Last Watermark:", last_watermark)

In [0]:
%sql
create schema if not exists banking.bronze


In [0]:
try:
    df = spark.read.jdbc(
        url=jdbc_url,
        table="(SELECT 1 AS x) AS t",
        properties=jdbc_properties
    )
    df.show()
except Exception as e:
    import traceback
    msg = str(e)
    print(msg[:2000])
    # Walk the cause chain
    cause = e
    while getattr(cause, "__cause__", None):
        cause = cause.__cause__
        print("CAUSED BY:", str(cause)[:1000])

In [0]:
try:
    spark.read.jdbc(
        url=jdbc_url,
        table="(SELECT 1 AS x) AS t",
        properties=jdbc_properties
    ).show()
except Exception as e:
    full = str(e)
    for line in full.splitlines():
        if any(k in line for k in ["Caused by", "SQLServerException", "Login failed",
                                   "not allowed to access", "TCP/IP", "timed out",
                                   "Cannot open", "SSL", "certificate"]):
            print(line[:500])

In [0]:
import socket
s = socket.socket()
s.settimeout(10)
try:
    s.connect(("atharvserver.database.windows.net", 1433))
    print("Port 1433 reachable")
except Exception as e:
    print("Cannot reach:", e)
finally:
    s.close()

In [0]:
# COMMAND ----------

# =====================================================
# 4️⃣ Read Source
# =====================================================

try:
    if source_system=="sqlserver":

        #Read connection JSON from secret
        secret_json=dbutils.secrets.get(
            scope="banking-scope",
            key="sqlserver-connection-json"
        )

        config=json.loads(secret_json)
        print(config['host'])
        print(config['port'])
        jdbc_url=f"jdbc:sqlserver://{config['host']}:{config['port']};databaseName={config['database']};encrypt=true;trustServerCertificate=true;"
        print(jdbc_url)

        jdbc_properties={
            "user":config['user'],
            "password":config['password'],
            "driver":config['driver']
        }

        #Build query
        if load_type in ["APPEND","MERGE"] and last_watermark:
            query=f"""
            (SELECT * FROM {source_schema}.{source_table}
            WHERE {watermark_column}>'{last_watermark}') AS src
            """
        else:
            query=f"(SELECT * FROM {source_schema}.{source_table}) AS src"
        
        source_df=spark.read.jdbc(
            url=jdbc_url,
            table=query,
            properties=jdbc_properties
        )
        print("source_df:",source_df)
    elif source_system=="blob":

        source_df=(spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option("cloudFiles.schemaLocation",
                f"/Volumes/banking/source/volume/_schema/{table_name}")
        .option("header","true")
        .load(source_path)
        )
        print("source_df:",source_df)
    else:
        raise ValueError("Unsupported source_system")


       # =====================================================
    # 5️⃣ Add insert_timestamp
    # =====================================================

    source_df=source_df.withColumn(
        "insert_timestamp",
        current_timestamp()
    )
    # =====================================================
    # 6️⃣ Write to Bronze (Append Only)
    # =====================================================

    if source_system=="blob":
        (
            source_df.writeStream
            .format("delta")
            .option(
                "checkpointLocation",
                f"/Volumes/banking/source/volume/_checkpoints/{table_name}"
            )            
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(bronze_table_fqn)
        )
        records_read=None
        print("written sucessfully")

    else:

        (
            source_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(bronze_table_fqn)
        )
        records_read=source_df.count()
    print("Source → Bronze Load Completed Successfully.")
    print("Watermark will be updated after Silver load.")

except Exception as e:
    import re
    full_error = str(e)

    # Pull out the useful root-cause lines
    root_lines = [l.strip() for l in full_error.splitlines()
                  if "Caused by" in l or "not allowed to access" in l or "Login failed" in l]
    root_cause = " | ".join(root_lines[:3]) or full_error[:1000]
    print("ROOT CAUSE:", root_cause)

    m = re.search(r"IP address '([\d\.]+)'", full_error)
    if m:
        print("BLOCKED IP (add this in Azure firewall):", m.group(1))

    safe_msg = root_cause[:4000].replace("'", "''")
    try:
        spark.sql(f"""
            UPDATE banking.metadata.pipeline_runs
            SET end_time = current_timestamp(),
                status = 'FAILED',
                error_message = '{safe_msg}'
            WHERE table_id = {table_id} AND run_id = {run_id}
        """)
    except Exception as log_err:
        print("Failed to write audit row:", log_err)
    raise
